# Goal-Based Warehouse Agent

A goal-based agent plans from `S` to `G` while avoiding shelves. Because each move costs one step, BFS returns a shortest path.

In [ ]:

from collections import deque
from dataclasses import dataclass

WAREHOUSE = [
    "#####################",
    "#S....#............G#",
    "#.##....##########..#",
    "#....##.............#",
    "#.######.###.#.###..#",
    "#........#..........#",
    "#####################",
]

MOVES = (
    ("Up", -1, 0),
    ("Down", 1, 0),
    ("Left", 0, -1),
    ("Right", 0, 1),
)


@dataclass
class Plan:
    positions: list[tuple[int, int]]
    actions: list[str]
    states_expanded: int

    @property
    def path_length(self) -> int:
        return len(self.actions)


class WarehouseAgent:
    """Goal-based agent that plans a collision-free route with BFS."""

    def __init__(self, grid):
        if not grid or len({len(row) for row in grid}) != 1:
            raise ValueError("Grid must be non-empty and rectangular.")

        self.grid = tuple(grid)
        self.rows = len(grid)
        self.cols = len(grid[0])
        self.start = self._find_unique("S")
        self.goal = self._find_unique("G")

    def _find_unique(self, symbol):
        locations = [
            (r, c)
            for r, row in enumerate(self.grid)
            for c, value in enumerate(row)
            if value == symbol
        ]
        if len(locations) != 1:
            raise ValueError(f"Expected exactly one {symbol}; found {len(locations)}.")
        return locations[0]

    def successors(self, state):
        """Generate legal actions and successor states."""
        row, col = state
        for action, dr, dc in MOVES:
            nr, nc = row + dr, col + dc
            if (
                0 <= nr < self.rows
                and 0 <= nc < self.cols
                and self.grid[nr][nc] != "#"
            ):
                yield action, (nr, nc)

    def find_path(self):
        """Breadth-first search. Unit move costs make the first found path shortest."""
        frontier = deque([self.start])
        parent = {self.start: (None, None)}
        expanded = 0

        while frontier:
            state = frontier.popleft()

            if state == self.goal:
                return self._reconstruct(parent, state, expanded)

            expanded += 1
            for action, nxt in self.successors(state):
                if nxt not in parent:
                    parent[nxt] = (state, action)
                    frontier.append(nxt)

        return None

    def _reconstruct(self, parent, goal, expanded):
        positions = []
        actions = []
        state = goal

        while state is not None:
            positions.append(state)
            previous, action = parent[state]
            if action is not None:
                actions.append(action)
            state = previous

        return Plan(
            positions=list(reversed(positions)),
            actions=list(reversed(actions)),
            states_expanded=expanded,
        )

    def draw_path(self, plan):
        canvas = [list(row) for row in self.grid]
        for r, c in plan.positions[1:-1]:
            canvas[r][c] = "*"
        return "\n".join("".join(row) for row in canvas)


def validate_plan(agent, plan):
    assert plan is not None, "A path should exist for the original warehouse."
    assert plan.positions[0] == agent.start
    assert plan.positions[-1] == agent.goal
    assert len(plan.positions) == plan.path_length + 1

    for previous, current in zip(plan.positions, plan.positions[1:]):
        r1, c1 = previous
        r2, c2 = current
        assert abs(r1 - r2) + abs(c1 - c2) == 1
        assert agent.grid[r2][c2] != "#"

    return True


def main():
    agent = WarehouseAgent(WAREHOUSE)
    plan = agent.find_path()
    validate_plan(agent, plan)

    print("Goal-based warehouse agent")
    print("--------------------------")
    print("Start:", agent.start)
    print("Goal :", agent.goal)
    print("Path length:", plan.path_length)
    print("States expanded:", plan.states_expanded)
    print("Actions:", " -> ".join(plan.actions))
    print("\nPath on warehouse:")
    print(agent.draw_path(plan))

    # Independent no-solution test.
    blocked = [
        "#####",
        "#S#G#",
        "#####",
    ]
    blocked_plan = WarehouseAgent(blocked).find_path()
    assert blocked_plan is None
    print("\nValidation: PASS")
    print("Blocked-grid test: PASS")




In [ ]:
main()


# Agents Lab — Short Answers

## Task 1: Understanding the problem

1. **Environment:** a deterministic two-dimensional warehouse grid containing free cells, shelf obstacles (`#`), a start state `S`, and a goal state `G`.
2. **Goal:** move the warehouse vehicle from `S` to `G` without entering an obstacle.
3. **Actions:** move one square **Up, Down, Left, or Right** when that destination square is valid.
4. **Information maintained:** the current position, warehouse map, goal position, the search frontier, visited states, and parent/action information used to reconstruct a path.
5. **Why goal-based:** the agent chooses actions by considering which future states can lead to an explicit goal. A simple reflex agent would only map the current percept directly to an action without planning a route.

### Think About It
BFS would still be correct on a warehouse twice as large, and with unit move costs it would still find a shortest path. However, its time and especially memory use can grow substantially because it may store a large frontier. An informed method such as A* can be more efficient when a suitable heuristic is available.

## Task 2: Agent design

```text
Warehouse map + current state
            |
            v
     Successor generator
   (legal U/D/L/R moves)
            |
            v
       BFS planner
(frontier + visited + parent)
            |
            v
       Goal test: G?
            |
            v
   Reconstruct action path
```

- **State:** `(row, column)` of the vehicle.
- **Goal test:** current state equals the coordinate of `G`.
- **Decision component:** breadth-first search.
- **Transition model:** each legal action changes the position by one grid square.

## Task 3: Prompt used

> Implement a goal-based warehouse navigation agent in Python. Represent the supplied warehouse as a 2-D grid, allow only Up/Down/Left/Right moves, never cross `#`, find a collision-free path from `S` to `G`, print the path or a no-path message, and use a search strategy appropriate for equal-cost moves. Include a visited set/parent structure and tests that verify the returned path is legal.

### Questions

1. **Did the generated program work?** Yes. The final generated implementation passed both a valid-path test and an explicit blocked/no-solution test.
2. **How could the prompt be improved if it failed?** Specify the exact grid symbols, legal actions, requirement to track visited states, expected output, and correctness tests.
3. **Search algorithm chosen:** Breadth-first search (BFS).
4. **Why BFS:** every legal move has equal cost, so BFS is complete on this finite grid and the first goal path it discovers has minimum number of moves.

## Validation / reflection

The implementation was not accepted merely because it ran. The returned path is checked to begin at `S`, end at `G`, move exactly one grid cell at a time, and never enter an obstacle. A second map verifies that the program correctly reports that no path exists.
